## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Spektrale Normalisierung: manuelle Implementierung ─────────────────────
class SpektralNormDense(tf.keras.layers.Layer):
    """
    Dense-Schicht mit spektraler Normalisierung.
    Normalisiert die Gewichtsmatrix durch ihren größten Singulärwert (σ_max),
    sodass die Lipschitz-Konstante ≤ 1 ist.
    """

    def __init__(self, einheiten, aktivierung=None, n_iter=1, **kwargs):
        super().__init__(**kwargs)
        self.einheiten   = einheiten
        self.aktivierung = tf.keras.activations.get(aktivierung)
        self.n_iter      = n_iter  # Potenz-Iterations-Schritte

    def build(self, eingabe_shape):
        self.W = self.add_weight(
            name="kernel",
            shape=(eingabe_shape[-1], self.einheiten),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="bias",
            shape=(self.einheiten,),
            initializer="zeros",
            trainable=True
        )
        # Rechte singuläre Vektor (für Potenz-Iteration)
        self.u = self.add_weight(
            name="u",
            shape=(1, self.einheiten),
            initializer="truncated_normal",
            trainable=False
        )
        super().build(eingabe_shape)

    def spektral_norm(self):
        """
        Schätzt σ_max (größten Singulärwert) via Potenz-Iteration:
        u ← W^T v / ||W^T v||   (Annäherung)
        σ ≈ u^T W^T v
        """
        W = tf.reshape(self.W, [-1, self.einheiten])
        u_hat = self.u

        for _ in range(self.n_iter):
            v_hat  = tf.math.l2_normalize(tf.matmul(u_hat, tf.transpose(W)))
            u_hat  = tf.math.l2_normalize(tf.matmul(v_hat, W))

        # Singulärwert: σ = v^T W u
        sigma = tf.squeeze(tf.matmul(tf.matmul(v_hat, W), tf.transpose(u_hat)))
        self.u.assign(u_hat)
        return sigma

    def call(self, eingabe, training=None):
        sigma  = self.spektral_norm()
        W_norm = self.W / sigma           # normalisierte Gewichte
        z      = tf.matmul(eingabe, W_norm) + self.b
        if self.aktivierung is not None:
            z = self.aktivierung(z)
        return z

    def get_config(self):
        config = super().get_config()
        config.update({"einheiten": self.einheiten,
                       "aktivierung": tf.keras.activations.serialize(self.aktivierung)})
        return config


# ── 2. Datensatz ──────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:5000].astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)         / 255.0
y_train = y_train[:5000]

# ── 3. Beide Modelle erstellen ────────────────────────────────────────────────
def standard_modell():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation="relu", input_shape=(784,)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10,  activation="softmax"),
    ], name="Standard_Dense")

def sn_modell():
    return tf.keras.Sequential([
        SpektralNormDense(256, aktivierung="relu", input_shape=(784,), name="sn_1"),
        SpektralNormDense(128, aktivierung="relu", name="sn_2"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name="Spektral_Norm_Dense")

m_std = standard_modell()
m_sn  = sn_modell()

for m in [m_std, m_sn]:
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# ── 4. Training ───────────────────────────────────────────────────────────────
EPOCHEN = 15
print("Trainiere Standard-Modell...")
h_std = m_std.fit(x_train, y_train, epochs=EPOCHEN, batch_size=64,
                   validation_split=0.1, verbose=0)

print("Trainiere Spektral-Norm-Modell...")
h_sn  = m_sn.fit(x_train, y_train, epochs=EPOCHEN, batch_size=64,
                   validation_split=0.1, verbose=0)

# ── 5. Singulärwert-Spektrum analysieren ──────────────────────────────────────
def singulärwerte_berechnen(schicht):
    """Berechnet alle Singulärwerte einer Dense-Schicht."""
    W = schicht.get_weights()[0]   # (n_in, n_out)
    _, s, _ = np.linalg.svd(W, full_matrices=False)
    return s

print("\n── Singulärwert-Analyse ──")
for name, m in [("Standard", m_std), ("SpektralNorm", m_sn)]:
    for i, schicht in enumerate(m.layers):
        if hasattr(schicht, "W") or isinstance(schicht, tf.keras.layers.Dense):
            W = schicht.get_weights()[0]
            s = np.linalg.svd(W, compute_uv=False)
            print(f"  {name} Layer {i}: σ_max={s[0]:.4f}, σ_min={s[-1]:.6f}, "
                  f"Konditionszahl={s[0]/s[-1]:.2f}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (a) Val-Accuracy
axes[0].plot(h_std.history["val_accuracy"], label="Standard",     linewidth=2)
axes[0].plot(h_sn.history["val_accuracy"],  label="SpektralNorm", linewidth=2)
axes[0].set_title("Validierungs-Genauigkeit")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True)

# (b) Singulärwert-Spektrum der ersten Schicht
for schicht_std, schicht_sn in zip(m_std.layers[:1], m_sn.layers[:1]):
    sv_std = singulärwerte_berechnen(schicht_std)
    sv_sn  = singulärwerte_berechnen(schicht_sn)
    axes[1].semilogy(sv_std, label="Standard",     marker=".", markersize=4)
    axes[1].semilogy(sv_sn,  label="SpektralNorm", marker=".", markersize=4)
axes[1].set_title("Singulärwerte (Schicht 1)")
axes[1].set_xlabel("Singulärwert-Index")
axes[1].set_ylabel("Singulärwert (log-Skala)")
axes[1].legend()
axes[1].grid(True)

# (c) Verteilung der Gewichte
for m, name, farbe in [(m_std, "Standard", "#e74c3c"),
                        (m_sn,  "SpektralNorm", "#2ecc71")]:
    w = m.layers[0].get_weights()[0].flatten()
    axes[2].hist(w, bins=50, alpha=0.6, color=farbe, label=name, density=True)
axes[2].set_title("Gewichtsverteilung (Schicht 1)")
axes[2].set_xlabel("Gewichtswert")
axes[2].set_ylabel("Dichte")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Spektrale Normalisierung: Gewichtsspektrum und Training", fontsize=13)
plt.tight_layout()
plt.savefig("E8_1_spektrale_normalisierung.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E8_1_spektrale_normalisierung.png")


## Exercise 2

**Dataset Used:** CIFAR-10 (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. CIFAR-10 Teilmenge laden ───────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train[:8000].astype("float32") / 255.0
x_test  = x_test[:2000].astype("float32")  / 255.0
y_train = y_train[:8000].flatten()
y_test  = y_test[:2000].flatten()
N_KLASSEN = 10

# ── 2. Label Smoothing ────────────────────────────────────────────────────────
GLAETTUNG = 0.1   # ε: "wieviel" von den harten One-Hot-Labels abweichen

def label_smoothing_verlust(y_wahr, y_pred, n_klassen=10, epsilon=0.1):
    """
    Weiche Ziel-Verteilung:  y_smooth = (1 - ε) * y_onehot + ε / K
    Cross-Entropy mit weichen Labels.
    """
    y_oh     = tf.one_hot(tf.cast(y_wahr, tf.int32), n_klassen)
    y_smooth = (1.0 - epsilon) * y_oh + epsilon / n_klassen
    log_pred = tf.math.log(y_pred + 1e-8)
    return -tf.reduce_mean(tf.reduce_sum(y_smooth * log_pred, axis=-1))

# ── 3. Mixup-Augmentierung ────────────────────────────────────────────────────
def mixup_batch(x, y, alpha=0.2):
    """
    Mixup: Interpoliert zwischen zwei Trainingsbeispielen.
    x_mix = λ * x_i + (1-λ) * x_j
    y_mix = λ * y_i + (1-λ) * y_j   (weiche Labels)
    λ ∼ Beta(α, α)
    """
    n = tf.shape(x)[0]
    lam = np.random.beta(alpha, alpha, size=n).astype("float32")
    lam = np.maximum(lam, 1 - lam)   # immer ≥ 0.5 für stabilere Interpolation
    idx = np.random.permutation(n)

    # Mixup auf Bilder anwenden
    lam_x = lam[:, np.newaxis, np.newaxis, np.newaxis]
    x_mix = lam_x * x + (1 - lam_x) * x[idx]

    # Weiche Labels erstellen
    y_oh  = tf.one_hot(y, N_KLASSEN).numpy()
    lam_y = lam[:, np.newaxis]
    y_mix = lam_y * y_oh + (1 - lam_y) * y_oh[idx]
    return x_mix.astype("float32"), y_mix.astype("float32")

# ── 4. Kleines CNN ────────────────────────────────────────────────────────────
def cnn_erstellen():
    m = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3,3), activation="relu",
                               padding="same", input_shape=(32,32,3)),
        tf.keras.layers.MaxPooling2D((2,2)),
        tf.keras.layers.Conv2D(64, (3,3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2,2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(N_KLASSEN, activation="softmax"),
    ])
    return m

# Optimizer und Verlustfunktionen
optimizer_std = tf.keras.optimizers.Adam(0.001)
optimizer_mix = tf.keras.optimizers.Adam(0.001)
std_verlust_fn = tf.keras.losses.SparseCategoricalCrossentropy()

# ── 5. Standard-Training (Baseline) ──────────────────────────────────────────
print("Trainiere Baseline (Standard-Training)...")
m_basis = cnn_erstellen()
m_basis.compile(optimizer="adam",
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])
h_basis = m_basis.fit(x_train, y_train, epochs=20, batch_size=64,
                       validation_data=(x_test, y_test), verbose=1)

# ── 6. Label Smoothing Training ───────────────────────────────────────────────
print("\nTrainiere mit Label Smoothing + Mixup (Custom Loop)...")
m_mix = cnn_erstellen()
EPOCHEN   = 20
BATCH_GR  = 64

ls_verlauf   = []
ls_val_accs  = []

for epoche in range(EPOCHEN):
    idx_all = np.random.permutation(len(x_train))
    epoch_losses = []

    for start in range(0, len(x_train), BATCH_GR):
        idx_b = idx_all[start:start+BATCH_GR]
        x_b   = x_train[idx_b]
        y_b   = y_train[idx_b]

        # Mixup anwenden
        x_mix, y_mix = mixup_batch(x_b, y_b, alpha=0.2)

        with tf.GradientTape() as tape:
            pred     = m_mix(x_mix, training=True)
            # Label-Smoothing-Verlust mit gemischten Labels
            log_pred = tf.math.log(pred + 1e-8)
            # Mixup Labels sind bereits soft → direkt Cross-Entropy
            loss     = -tf.reduce_mean(
                tf.reduce_sum(tf.cast(y_mix, tf.float32) * log_pred, axis=-1)
            )
        grads = tape.gradient(loss, m_mix.trainable_variables)
        optimizer_mix.apply_gradients(zip(grads, m_mix.trainable_variables))
        epoch_losses.append(float(loss))

    # Validierung
    val_pred = m_mix(x_test, training=False)
    val_acc  = np.mean(np.argmax(val_pred.numpy(), axis=1) == y_test)
    ls_verlauf.append(np.mean(epoch_losses))
    ls_val_accs.append(val_acc)
    print(f"Epoche {epoche+1:2d}/{EPOCHEN} | Loss: {np.mean(epoch_losses):.4f} | "
          f"Val-Acc: {val_acc:.4f}")

# ── 7. Vergleich ──────────────────────────────────────────────────────────────
acc_basis = m_basis.evaluate(x_test, y_test, verbose=0)[1]
acc_mix   = float(np.mean(np.argmax(m_mix.predict(x_test, verbose=0), axis=1) == y_test))
print(f"\nTest-Genauigkeit Baseline:   {acc_basis:.4f}")
print(f"Test-Genauigkeit LS+Mixup:   {acc_mix:.4f}")

# ── 8. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(h_basis.history["val_accuracy"], label="Baseline", linewidth=2)
axes[0].plot(ls_val_accs,                      label="LS + Mixup", linewidth=2)
axes[0].set_title("Validierungs-Genauigkeit")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True)

# Mixup-Beispiele
fig2, ax2 = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    ax2[0, i].imshow(x_train[i])
    ax2[0, i].set_title(f"Orig {y_train[i]}")
    ax2[0, i].axis("off")
x_m, y_m = mixup_batch(x_train[:5], y_train[:5], alpha=0.4)
for i in range(5):
    ax2[1, i].imshow(np.clip(x_m[i], 0, 1))
    klasse = np.argmax(y_m[i])
    ax2[1, i].set_title(f"Mixup ~{klasse}")
    ax2[1, i].axis("off")
plt.suptitle("Mixup Augmentierung: Original vs. gemischt", fontsize=12)
plt.tight_layout()
plt.savefig("E8_2_mixup_beispiele.png", dpi=100)
plt.show()

axes[1].bar(["Baseline", "LS + Mixup"], [acc_basis, acc_mix],
            color=["#e74c3c", "#2ecc71"], edgecolor="black")
axes[1].set_title("Test-Genauigkeit Vergleich")
axes[1].set_ylabel("Genauigkeit")
axes[1].set_ylim(0, 1)
for i, a in enumerate([acc_basis, acc_mix]):
    axes[1].text(i, a + 0.01, f"{a:.4f}", ha="center")
axes[1].grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("E8_2_label_smoothing_mixup.png", dpi=100)
plt.show()
print("Diagramme gespeichert: E8_2_*.png")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 8: Regularisierung (Dropout, BatchNorm)
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Stochastic Depth: Schicht zufällig überspringen ───────────────────────
class StochasticDepthBlock(tf.keras.layers.Layer):
    """
    Stochastic Depth: Während des Trainings wird jede Residual-Schicht
    mit Wahrscheinlichkeit (1 - survival_prob) komplett übersprungen.
    Beim Inference: gewichtete Skalierung durch survival_prob.
    """

    def __init__(self, einheiten, survival_prob=0.8, **kwargs):
        super().__init__(**kwargs)
        self.survival_prob = survival_prob
        self.dense_1 = tf.keras.layers.Dense(einheiten, activation="relu")
        self.dense_2 = tf.keras.layers.Dense(einheiten)
        self.relu    = tf.keras.layers.Activation("relu")

    def call(self, eingabe, training=None):
        if training:
            # Bernoulli-Entscheidung: Schicht überspringen?
            zufallswert = tf.random.uniform(shape=())
            if zufallswert > self.survival_prob:
                # Schicht überspringen → direkte Residual-Verbindung
                return self.relu(eingabe)
            else:
                # Schicht ausführen
                x = self.dense_1(eingabe)
                x = self.dense_2(x)
                return self.relu(x + eingabe)
        else:
            # Inference: Schicht wird mit survival_prob skaliert
            x = self.dense_1(eingabe)
            x = self.dense_2(x)
            return self.relu(self.survival_prob * x + eingabe)

    def get_config(self):
        config = super().get_config()
        config.update({"einheiten": self.dense_1.units,
                        "survival_prob": self.survival_prob})
        return config


# ── 2. Linearer Abfall der Überlebenswahrscheinlichkeit ──────────────────────
def survival_probs_berechnen(n_schichten, p_min=0.5):
    """
    Lineare Abnahme: erste Schicht hat p=1.0, letzte hat p=p_min.
    (Tiefere Schichten werden häufiger übersprungen.)
    """
    return [1.0 - (1.0 - p_min) * i / (n_schichten - 1)
            for i in range(n_schichten)]

N_SCHICHTEN    = 10
EINHEITEN      = 64
P_MIN          = 0.5     # Überlebenswahrscheinlichkeit der letzten Schicht
probs          = survival_probs_berechnen(N_SCHICHTEN, P_MIN)

print(f"Überlebenswahrscheinlichkeiten ({N_SCHICHTEN} Schichten):")
for i, p in enumerate(probs):
    print(f"  Schicht {i+1:2d}: {p:.3f}")

# ── 3. MNIST-Daten ────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:8000].astype("float32").reshape(-1, 784) / 255.0
x_test  = x_test.astype("float32").reshape(-1, 784)         / 255.0
y_train = y_train[:8000]

# ── 4. Modell MIT Stochastic Depth ────────────────────────────────────────────
def stochastic_depth_modell():
    eingabe = tf.keras.Input(shape=(784,))
    x = tf.keras.layers.Dense(EINHEITEN, activation="relu")(eingabe)

    for i in range(N_SCHICHTEN):
        x = StochasticDepthBlock(
            EINHEITEN, survival_prob=probs[i], name=f"sd_block_{i+1}"
        )(x)

    ausgabe = tf.keras.layers.Dense(10, activation="softmax")(x)
    return tf.keras.Model(eingabe, ausgabe, name="Stochastic_Depth_Netz")

# ── 5. Standard-Modell (ohne Stochastic Depth, gleiche Tiefe) ─────────────────
def standard_tiefes_modell():
    schichten = [tf.keras.layers.Dense(EINHEITEN, activation="relu",
                                        input_shape=(784,))]
    for _ in range(N_SCHICHTEN):
        schichten.append(tf.keras.layers.Dense(EINHEITEN, activation="relu"))
    schichten.append(tf.keras.layers.Dense(10, activation="softmax"))
    return tf.keras.Sequential(schichten, name="Standard_Tiefes_Netz")

m_sd  = stochastic_depth_modell()
m_std = standard_tiefes_modell()

for m in [m_sd, m_std]:
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

print(f"\nStochastic-Depth-Modell – Parameter: {m_sd.count_params():,}")
print(f"Standard-Modell         – Parameter: {m_std.count_params():,}")

# ── 6. Training ───────────────────────────────────────────────────────────────
EPOCHEN = 25
print("\nTrainiere Standard-Modell...")
h_std = m_std.fit(x_train, y_train, epochs=EPOCHEN, batch_size=64,
                   validation_split=0.1, verbose=0)

print("Trainiere Stochastic-Depth-Modell...")
h_sd  = m_sd.fit(x_train, y_train, epochs=EPOCHEN, batch_size=64,
                  validation_split=0.1, verbose=0)

# ── 7. Ergebnisse ─────────────────────────────────────────────────────────────
acc_std = m_std.evaluate(x_test, y_test, verbose=0)[1]
acc_sd  = m_sd.evaluate(x_test, y_test,  verbose=0)[1]
print(f"\nTest-Genauigkeit Standard:         {acc_std:.4f}")
print(f"Test-Genauigkeit Stochastic Depth: {acc_sd:.4f}")

gap_std = (h_std.history["accuracy"][-1] - h_std.history["val_accuracy"][-1])
gap_sd  = (h_sd.history["accuracy"][-1]  - h_sd.history["val_accuracy"][-1])
print(f"Overfitting-Gap Standard:          {gap_std:.4f}")
print(f"Overfitting-Gap Stochastic Depth:  {gap_sd:.4f}")

# ── 8. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Überleben-Wahrscheinlichkeiten
axes[0].bar(range(1, N_SCHICHTEN+1), probs, color="steelblue", edgecolor="black")
axes[0].set_title("Überlebenswahrscheinlichkeit\npro Schicht (Stochastic Depth)")
axes[0].set_xlabel("Schicht")
axes[0].set_ylabel("p (Survival Probability)")
axes[0].set_ylim(0, 1.05)
axes[0].axhline(1.0, color="red", linestyle="--", alpha=0.5, label="Immer aktiv")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Val-Accuracy
axes[1].plot(h_std.history["val_accuracy"], label="Standard",         linewidth=2)
axes[1].plot(h_sd.history["val_accuracy"],  label="Stochastic Depth", linewidth=2)
axes[1].set_title("Validierungs-Genauigkeit")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[1].grid(True)

# Overfitting-Gap
kategorien = ["Standard", "Stochastic\nDepth"]
luecken    = [gap_std, gap_sd]
farben     = ["#e74c3c", "#2ecc71"]
axes[2].bar(kategorien, luecken, color=farben, edgecolor="black")
axes[2].set_title("Overfitting-Gap\n(Train-Acc − Val-Acc, letzte Epoche)")
axes[2].set_ylabel("Gap")
for i, g in enumerate(luecken):
    axes[2].text(i, g + 0.002, f"{g:.4f}", ha="center")
axes[2].grid(True, axis="y", alpha=0.3)

plt.suptitle(f"Stochastic Depth: {N_SCHICHTEN} Schichten, p_min={P_MIN}", fontsize=13)
plt.tight_layout()
plt.savefig("E8_3_stochastic_depth.png", dpi=100)
plt.show()
print("Diagramm gespeichert: E8_3_stochastic_depth.png")
